# 残差连接

- 发现随着网络深度增加，反而误差增加，因为深层网络难以优化。即使之前的网络已经是最优解了，模型很难在更深的网络中学到一个恒等映射，让loss不增加；
- 网络增加一层后，输出由两部分构成，一部分是输入x，一部分是新增加层对x做的修改，叫残差部分，用F(x)表示。增加的这一层最终输出的特征是F(x)+x，期望F(x)+x = H(x)；
- 使模型更容易学到恒等映射，因为只需要学习到输出接近于0；
- 真实映射 ≈ 输入 + 小扰动，这个小扰动就是定义的残差F(x)，更容易学习；
- 梯度反向传播：$\frac{\partial Loss}{\partial x}=\frac{\partial Loss}{\partial y} \cdot (1 + \frac{\partial F}{\partial x})$，即使在深层网络中$\frac{\partial F}{\partial x} \to 0$，梯度仍然存在，解决了梯度消失和深层训练困难的问题；
- 在已有图上修改，显然比 从零开始画一张图 更简单；
- Residual Block分为 **BasicBlock** 和 **Bottleneck**：
    - BasicBlock：用于ResNet-18/34，由两个3*3卷积层组成
    - Bottleneck：用于ResNet-50/101/152，由一个1*1卷积层、一个3*3卷积层、一个1*1卷积层组成

In [5]:
import torch
import torch.nn as nn

BasicBlock

- 在Conv3阶段开始，由于输入残差块和输出的特征尺度不一样，所以在其捷径连接上要加入一个1\*1,s=2的卷积块。确实跳过了部分特征，损失了一定的信息。但是残差连接必须保证特征图的长宽以及通道数都一致才能进行按位相加，而1\*1的卷积、步长为2是达到这一点（长宽减半，通道数加倍）计算量最小的实现，符合残差连接设计的哲学（捷径连接尽量不产生额外的计算量）。
- 确实会丢失信息，但信息主要是靠主通道进行传递的。特征信息主要蕴藏在通道数里面

In [6]:
class BasicBlock(nn.Module):
    """
    Standard ResNet BasicBlock
    2个3*3卷积，每个卷积后跟 BN 和 ReLU。
    若输入和输出的维度不一致，则在捷径连接上使用 1*1卷积 s=2
    """

    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super(BasicBlock, self).__init__()
        # 第一个Residual Block的卷积层可能输入和输出通道数不一致：
        # 对于Conv2，第一个Residual Block的卷积层输入和输出通道数一致。
        # 对于Conv3-5的第一个Residual Block：输入输出通道数不一致，则stride设置为2，达到同时减半高宽，翻倍通道数。
        # 第二个Residual Block的卷积层输入和输出通道数一致，stride设置为1，保持特征图尺寸一致。
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride,
                               padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        # 第二个卷积层的输入和输出通道数都是out_channels,stride=1,保证输入和输出特征图尺寸一致。
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1,
                               padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        # 如果输入和输出特征图尺寸不一致，需要下采样，即调用stride=2的1×1卷积进行特征图尺寸的调节。
        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out

Bottleneck：

- 用于更深的ResNet网络结构中，为了减少计算量

In [7]:
class Bottleneck(nn.Module):
    # Bottleneck block for deeper ResNet
    # 使用 1*1 降维 → 3*3卷积 → 1*1恢复通道数
    expansion = 4   # bottleneck的输出通道数会相对于输入扩展的倍数；BasicBlock中为 1

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super(Bottleneck, self).__init__()
        mid_channels = out_channels
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(mid_channels)
        self.conv2 = nn.Conv2d(mid_channels, mid_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(mid_channels)
        self.conv3 = nn.Conv2d(mid_channels, out_channels * self.expansion,
                               kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out

ResNet 实现

In [10]:
class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=1000):
        super(ResNet, self).__init__()
        self.in_channels = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, out_channels, blocks, stride=1):
        downsample = None
        if stride != 1 or self.in_channels != out_channels * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels * block.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * block.expansion),
            )

        layers = [block(self.in_channels, out_channels, stride, downsample)]
        self.in_channels = out_channels * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.in_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1) # 保留 batch 维度
        x = self.fc(x)
        return x

In [11]:
def res_net50(num_classes=1000):
    return ResNet(Bottleneck, [3, 4, 6, 3], num_classes=num_classes)

resnet50 = res_net50(num_classes=1000)
print(resnet50)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [12]:
def res_net18(num_classes=1000):
    return ResNet(Bottleneck, [2, 2, 2, 2], num_classes=num_classes)

resnet18 = res_net18(num_classes=1000)
print(resnet18)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 